# 05.28 - Advanced Ensembling Strategies

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Combining multiple models often outperforms any single model. Advanced ensembling goes beyond simple averaging.

## 2. Why Does This Matter?

Ensembling is the secret weapon in Kaggle competitions and production ML.

## 3. Prerequisites

- 05.06: Random forests
- 05.07: Gradient boosting

## 4. Learning Objectives

- Implement stacking, blending, and voting
- Use diverse base models
- Optimize ensemble weights

## 5. Mental Model

Bagging: parallel models, average.
Boosting: sequential models, correct errors.
Stacking: train meta-learner on base model outputs.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Libraries loaded.")

Libraries loaded.


## 6. Base Models

In [2]:
wine = load_wine()
X, y = wine.data, wine.target

models = {
    "RF": RandomForestClassifier(n_estimators=100, random_state=42),
    "GB": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "LR": Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, random_state=42))]),
    "SVM": Pipeline([("scaler", StandardScaler()), ("clf", SVC(probability=True, random_state=42))])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv)
    print(name + ": " + str(round(scores.mean(), 4)) + " (+/- " + str(round(scores.std(), 4)) + ")")

RF: 0.9775 (+/- 0.0213)


GB: 0.9214 (+/- 0.0328)
LR: 0.9833 (+/- 0.0136)


SVM: 0.9833 (+/- 0.0222)


## 7. Voting Classifier

In [3]:
voting = VotingClassifier(
    estimators=[(n, m) for n, m in models.items()],
    voting="soft"
)
scores_voting = cross_val_score(voting, X, y, cv=cv)
print("Voting: " + str(round(scores_voting.mean(), 4)) + " (+/- " + str(round(scores_voting.std(), 4)) + ")")

Voting: 0.9944 (+/- 0.0111)


## 8. Stacking

In [4]:
from sklearn.ensemble import StackingClassifier

stacking = StackingClassifier(
    estimators=[(n, m) for n, m in models.items()],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)
scores_stacking = cross_val_score(stacking, X, y, cv=cv)
print("Stacking: " + str(round(scores_stacking.mean(), 4)) + " (+/- " + str(round(scores_stacking.std(), 4)) + ")")

Stacking: 0.9944 (+/- 0.0111)


## 9. Custom Blending

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

base_preds = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict_proba(X_test)
    base_preds.append(pred)

avg_pred = np.mean(base_preds, axis=0)
accuracy = np.mean(avg_pred.argmax(axis=1) == y_test)
print("Custom blend accuracy:", round(accuracy, 4))

weights = [0.3, 0.3, 0.2, 0.2]
weighted_pred = np.average(base_preds, axis=0, weights=weights)
accuracy_w = np.mean(weighted_pred.argmax(axis=1) == y_test)
print("Weighted blend accuracy:", round(accuracy_w, 4))

Custom blend accuracy: 1.0
Weighted blend accuracy: 1.0


## 10. Ensemble Comparison

In [6]:
results = [
    {"Method": "Individual Best", "Accuracy": max([cross_val_score(m, X, y, cv=cv).mean() for m in models.values()])},
    {"Method": "Voting", "Accuracy": scores_voting.mean()},
    {"Method": "Stacking", "Accuracy": scores_stacking.mean()}
]
print(pd.DataFrame(results).to_string(index=False))

         Method  Accuracy
Individual Best  0.983333
         Voting  0.994444
       Stacking  0.994444


## 11. Common Mistakes

1. Using correlated models
2. Overfitting the meta-learner
3. Too many base models
4. Ignoring inference time

## 12. Coding Exercises

### Exercise 1: Custom Ensemble
Build a custom ensemble with weighted voting.

### Exercise 2: Blend with XGBoost
Add XGBoost to the ensemble.

In [7]:
# EXERCISE 1: Custom weighted ensemble
print("Exercise: Optimize ensemble weights.")

Exercise: Optimize ensemble weights.


In [8]:
# EXERCISE 2: Add XGBoost
try:
    from xgboost import XGBClassifier
    print("Exercise: Add XGBoost to ensemble.")
except ImportError:
    print("XGBoost not installed.")

Exercise: Add XGBoost to ensemble.


## 13. Closed-Book Recall

1. What is stacking?
2. What is the difference between voting and stacking?
3. When does ensembling help most?

## 14. Teach-Back Questions

Explain stacking vs voting. How diversity improves ensembles.

## 15. Summary

Advanced ensembling combines diverse models. Voting averages predictions. Stacking trains a meta-learner. Custom blending gives full control.

## 16. Further Experiment

1. Try blending with neural networks.
2. Optimize ensemble weights with Optuna.
3. Measure inference time tradeoffs.

## Verification Status
```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: [numpy, pandas, matplotlib, scikit-learn]
OUTPUTS: PASS
LAST VERIFIED: 2026-08-30
```